In [ ]:
import warnings
from pathlib import Path

from statsmodels.tools.sm_exceptions import InterpolationWarning

from input import input
import config
from model import generics, hybrid_system_exp, grid_search_exp
from model.feature_selection import TimeSeriesFeatureSelector
from metrics import metrics
import numpy as np

from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
import pandas as pd

%load_ext autoreload
%autoreload 2

In [ ]:
# === CELULA DE CONFIGURACAO -- Tarefa 3.2 (fluxo notebook-only) ===
# Edite aqui para uma nova rodada: series, experiment_id, grid. Ver
# RUNBOOK.md Secao 1 (series/lag_size) e Secao 3 (convencao de experiment_id).
#
# Tarefa 3 do PLANO_ARQUITETURA.md: integracao do TimeSeriesFeatureSelector
# via sklearn.Pipeline. strategy fixa nesta execucao ('mutual_info' -- filtro
# por teoria da informacao, promove o experimento exploratorio ja existente
# em src/test_mutual_information.py para a arquitetura formal). k varia como
# grid (selector__k abaixo). A transformacao de y (MinMaxScaler/KPSS) continua
# fora do Pipeline, dentro de Additive/generics.format_forecats (Secao 1.4 do
# PLANO_ARQUITETURA.md).
model = Pipeline([
    ('selector', TimeSeriesFeatureSelector(strategy='mutual_info')),
    ('estimator', MLPRegressor(activation='logistic', solver='lbfgs')),
])

# Series a rodar nesta execucao (FS_DEV_SERIES por padrao -- tests/model/conftest.py).
# austres.txt resolve para 1 unico lag via lag_size='auto' (RUNBOOK.md Secao 1) --
# avaliar se faz sentido incluir antes de comparar metodos de FS entre si.
fs_series_list = ['airlines.txt', 'austres.txt', 'coloradoRiver.txt', 'sunspot.txt']

# experiment_id novo e explicito (Secao 3.2 do CLAUDE.md) -- nunca reaproveitar.
experiment_id = 'chamados_v4_fs_mutualinfo'
# Caminhos derivados de experiment_id, computados uma vez aqui e reusados nas
# celulas 3/5/6 (achado de code-review da Tarefa 3.2 -- evita reconstruir o
# mesmo Path 3x).
experiment_dir = Path(config.MODEL_DATA_PATH) / experiment_id
experiment_dir_results = Path(config.ROOT_PATH) / 'results' / experiment_id

# Sufixo sem underscore -- extract_series_name_from_path/model_name em
# metrics.py continuam funcionando sem nenhuma mudanca (ver Tarefa 3, Parte B).
model_name = 'amv1mutualinfo'
normalize = True
force = False
model_exec = 10

experiment_params = {
    'linear_model_name': '1arima',
    'diff_kpss': False,
    'horizon': 1,
}

# Convencao selector__*/estimator__* (nativa do sklearn): GridSearch usa
# clone(model).set_params(**params), que resolve essas chaves automaticamente
# via Pipeline.get_params(deep=True) -- nenhuma mudanca em grid_search_exp.py.
model_parameters = {
    'selector__k': [1, 5, 9, 15, 20],
    'estimator__hidden_layer_sizes': [10, 20, 50],
    'estimator__max_iter': [1000],
   }

In [ ]:
# Sanity-check exigido pela Tarefa 2: confirma que Pipeline.get_params(deep=True)
# expoe as chaves selector__*/estimator__* que GridSearch (ParameterGrid +
# clone().set_params()) vai usar, antes de rodar o grid completo.
params = model.get_params(deep=True)
required_keys = {'selector__strategy', 'selector__k', 'estimator__hidden_layer_sizes', 'estimator__max_iter'}
missing = required_keys - params.keys()
assert not missing, f"get_params(deep=True) nao expos chaves esperadas: {missing}"
print("OK -- Pipeline.get_params(deep=True) expoe:", sorted(required_keys))

# Checagem de identidade (achado de code-review da Tarefa 3.2): trava que a
# 'strategy' do seletor (celula 1) e o experiment_id/model_name declarados
# na MESMA celula continuam consistentes entre si -- protege contra editar
# 'strategy' na celula de configuracao por engano e deixar experiment_id/
# model_name (e portanto o .pkl/CSV gerado) mislabeled com o nome de outra
# estrategia.
strategy_slug = model.named_steps['selector'].strategy.replace('_', '')
assert strategy_slug in experiment_id, (
    f"strategy={model.named_steps['selector'].strategy!r} nao aparece em "
    f"experiment_id={experiment_id!r} -- confirme que voce nao mudou 'strategy' "
    "na celula de configuracao sem atualizar experiment_id/model_name."
)
print(f"OK -- strategy={model.named_steps['selector'].strategy!r} consistente com experiment_id={experiment_id!r}")

In [ ]:
# Copia o ARIMA pre-treinado (mesmo modelo, nao retreina) para o novo
# experiment_id -- Additive precisa dele sob o MESMO experiment_id da variante
# FS (ver RUNBOOK.md Secao 3). Usa copy_pretrained_linear_model (src/utils/),
# que por baixo usa generics.format_names() -- o mesmo helper que Additive/
# input_linear_info usam para localizar esse .pkl -- e le o nome do modelo
# linear de experiment_params['linear_model_name'] (celula 1), em vez de um
# literal cravado (achado de code-review da Tarefa 3.2).
from utils.copy_pretrained_linear_model import copy_pretrained_linear_model

copy_pretrained_linear_model(
    source_experiment_id='chamados',
    dest_experiment_id=experiment_id,
    series_list=fs_series_list,
    linear_model_name=experiment_params['linear_model_name'],
)

In [ ]:
# Chamado direto via GridSearch(...).execution() por serie, em vez de
# grid_seach_multiple_bases(), para nao depender/mutar config.BASE_NAME_LIST e
# sem tocar grid_search_exp.py. use_val_slipt_for_prev=True e explicito porque
# o default de GridSearch (False) diverge do default do wrapper
# grid_seach_multiple_bases (True) -- sem isso, o refit final perderia o
# val_size e os .pkl ficariam sem val_metrics, quebrando a paridade com o
# baseline original (amv1 em chamados/).
for base_name in fs_series_list:
    print(base_name)
    exec_gs = grid_search_exp.GridSearch(
        hybrid_system_exp.Additive,
        model,
        model_parameters,
        experiment_id,
        base_name,
        model_name,
        force,
        normalize,
        experiment_params,
        model_exec=model_exec,
        use_val_slipt_for_prev=True,
    )
    exec_gs.execution()

In [ ]:
# Gera o CSV de metricas (agregado + detalhado) direto no notebook -- mesma
# funcao usada pela CLI (src/utils/export_metrics_to_csv.py), so o ponto de
# entrada muda (Tarefa 3.2, fluxo notebook-only). Equivale ao Passo 2 do
# RUNBOOK.md Secao 8b.
from utils.export_metrics_to_csv import run_export_metrics_to_csv

metrics_output = experiment_dir_results / 'metrics.csv'
df_metrics = run_export_metrics_to_csv(
    experiment_dir,
    metrics_output,
    detail=True,
)
df_metrics

In [ ]:
# Gera o CSV de features selecionadas (agregado + detalhado) direto no
# notebook -- mesma funcao usada pela CLI (src/utils/export_selected_features.py),
# so o ponto de entrada muda (Tarefa 3.2, fluxo notebook-only). Equivale ao
# Passo 2 do RUNBOOK.md Secao 8b.
from utils.export_selected_features import run_export_selected_features

features_output = experiment_dir_results / 'selected_features.csv'
df_features = run_export_selected_features(
    experiment_dir,
    features_output,
    detail=True,
)
df_features